# Lab 8 — Connect the five stages
### EP0418 · Topic 8 · 2 practical hours

---
You have built every piece separately. Today you connect them — and the joins are
where systems actually break.

## The method: skeleton first

**Do not** connect five finished components and hope. Build the whole chain with
everything faked, get it running end to end, then replace one stage at a time.

If you connect five untested things and it fails, you have five suspects.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see a few lines saying *fetched ...* and nothing in red; that means the course files and tools are loaded. Every lab starts this way. **To write in a text box** like *Strangest number:* further down, double-click the box, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

## Task 1 — The skeleton, entirely faked

Get this running in the first twenty minutes. It does nothing useful. That is the
point.

In [ ]:
def ingest():
    """Stage 1 — later: read your real data."""
    return pd.DataFrame({"timestamp": pd.date_range("2026-06-01", periods=10, freq="h"),
                         "value": range(10)})

def clean(df):
    """Stage 2 — later: your real cleaning."""
    return df

def analyse(df):
    """Stage 3 — later: your detector or forecast."""
    return df.assign(flag=False)

def decide(df):
    """Stage 4 — later: your rule, debounced and rate-limited."""
    return df[df.flag]

def respond(events):
    """Stage 5 — later: your real notification."""
    print(f"would send {len(events)} alert(s)")
    return [f"alert {i}" for i in range(len(events))]

def pipeline():
    df = ingest()
    print(f"  ingest  -> {len(df):,} rows")
    df = clean(df)
    print(f"  clean   -> {len(df):,} rows")
    df = analyse(df)
    print(f"  analyse -> {len(df):,} rows")
    events = decide(df)
    print(f"  decide  -> {len(events):,} events")
    return respond(events)

sent = pipeline()

### Note the row count printed at every stage

That is not decoration. **"Zero alerts" and "a stage is silently producing
nothing" look identical from outside.** Printing the shape at each stage is how
you tell them apart — in this lab and in your CA3 report.

## Task 2 — Replace one stage at a time

Swap in your real `ingest`. Run the whole pipeline. Still works? Move to `clean`.

**Run the full chain after every single swap.** When it breaks you will know
exactly which change broke it.

### Turning your earlier cells into a function

Your Lab 4 cleaning was written as loose cells on the classroom data. A stage is
the same lines wrapped in `def`, taking a table in and giving a table back:

```python
def clean(df):
    df = df.copy()
    df.loc[df.humidity_pct == -999, "humidity_pct"] = np.nan     # your Lab 4 lines
    df = df.drop_duplicates("timestamp", keep="last").sort_values("timestamp")
    return df
```

Use your group's project data and columns. The starter pack for that track shows
how to load it; the timestamp column and cadence differ by track:

| Track | File | Time column | Cadence |
|---|---|---|---|
| 1 Smart Money | `smart_money_ohlcv.csv` | `Date` | daily |
| 2 Smart Building | `smart_building_sensors.csv` | `timestamp` | 5 min |
| 3 Smart Farm | `sg_rainfall_daily.csv` | `date` | daily |
| 4 Smart City | `sg_taxi_hourly.csv` | `timestamp` | hourly |
| 5 Smart Self | `smart_self_daily.csv` | `date` | daily |

If your data has no `-999` sentinel or no duplicates, `clean()` may be short.
It must still return the table.

### Or start from the track kit

`track_kit.py` (beside this notebook, upload it with `ep0418.py`) has a
ready-made `ingest()` and `clean()` for every track, each handling that file's
documented faults. Copy the pair for your data, **read every line**, then
adapt it. Every kit `ingest()` returns a table with a `timestamp` column, so the
rest of your pipeline can assume that name.

In [ ]:
fetch("track_kit.py")
# Uncomment the line for your data. Then open track_kit.py and read the two
# functions you just imported: you are responsible for what they do.
# from track_kit import ingest_smart_money    as ingest, clean_smart_money    as clean   # Track 1
# from track_kit import ingest_smart_building as ingest, clean_smart_building as clean   # Track 2
# from track_kit import ingest_smart_farm     as ingest, clean_smart_farm     as clean   # Track 3
# from track_kit import ingest_smart_city     as ingest, clean_smart_city     as clean   # Track 4
# from track_kit import ingest_smart_self     as ingest, clean_smart_self     as clean   # Track 5
# from track_kit import ingest_free_form      as ingest, clean_free_form      as clean   # Track 6
# sent = pipeline()

In [ ]:
# TODO(you): replace ingest() with your real data loading, then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace clean(), then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace analyse(), then re-run the whole chain
...
sent = pipeline()

In [ ]:
# TODO(you): replace decide() and respond(), then re-run the whole chain
...
sent = pipeline()

## Task 3 — Verify the whole chain

The contract below checks the things that break at the joins.

In [ ]:
raw_df   = ingest()
clean_df = clean(raw_df)

# TODO(you): explain any change in row count between the two stages
EXPLANATION = "TODO(you): explain any change in row count through your pipeline"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw_df, clean_df,
    names=["ingest", "clean"],
    explain=EXPLANATION,
)
check.response_fired(sent, minimum=1)
check.report()

show.stages({"ingest": len(raw_df), "clean": len(clean_df)}, title="Rows at each stage boundary")

**What you should see:** two bars, and a difference between them that your
`EXPLANATION` accounts for line by line. If the bars are equal and you cleaned
anything, the cleaning did not run on the data that reached this stage.

## Task 4 — Draw it

Five boxes, four arrows. On each arrow write what passes between the stages —
**what shape, what cadence, what columns**.

This diagram goes into your CA3 report, so it is not throwaway work.

---
## Predict: Which day will be wettest? (10 minutes)

*The Prediction League, week 7. Teams of two or three, one team name all semester.*

**The question:** Which day of the target week will have the most rain across the island? Answer 1 for Monday through 7 for Sunday.
One number per team. **Teams:** turn to one or two neighbours now and agree a
name; keep the same team and name every week. **The scoreboard** is at
https://stanleysp-eng.github.io/EP0418-labs/ (open it on a phone): it shows the
target week's dates, every team's number the moment it is sent, and the real
number when the week is over. Nearest team wins the week.

**Step 1. Run the cell below.** It prints two candidate numbers, A and B, and the
exact question to ask Gemini.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])
wk = rain.set_index("date").rain_mm_mean
wettest = wk.groupby(wk.index.to_period("W")).idxmax().dt.weekday + 1
by_day = wk.groupby(wk.index.weekday + 1).mean()
A = int(wettest.mode().iloc[0])                # the weekday that was wettest most often
B = int(by_day.idxmax())                       # the weekday with the highest average rain
print(f"A. the weekday most often wettest : {A}   (1 = Mon ... 7 = Sun)   <- the baseline on the board")
print(f"B. the weekday with most rain on average : {B}")
print(f"   (wettest weekday of each week in the file: {wettest.tolist()})")

print()
print("Step 2. Open Gemini (spark icon, bottom right), press + for a new chat, and paste this:")
print()
print(f"I am new to this. Answer in three plain sentences, no headings, no code. I must predict which day of the target week will be the wettest, as a number from 1 (Monday) to 7 (Sunday). "
      f"One estimate says {A:.0f}, based on last time; another says {B:.0f}, based on what is typical in my data. "
      "Which should I use, or a number in between, and why might it still be wrong?")

**Step 2. Ask Gemini.** Paste the printed question into the chat and press Enter.
Read the three sentences. It will pick A, B or something between and give a reason.
Ignore the blue suggestion chips under its answer. If it writes an essay, type
*simpler, three sentences*.

**Step 3. Type your number and send it.** Talk it over with your team: Gemini's
pick, or your own. Put it in the cell below, with your team name and one line on
why, and run the cell. Filled in, it looks like this:

```
TEAM = "Team Rocket"
MY_PREDICTION = 4
METHOD = "Gemini said B; we nudged it because ..."
```

The message *Entered: Team Rocket predicts ...* confirms it. Open the scoreboard
(https://stanleysp-eng.github.io/EP0418-labs/) and look for your team under
"Prediction League" within a minute. Nobody is marked on the method; only reality
marks the number.

In [ ]:
TEAM = ""                 # TODO(you): your team name
MY_PREDICTION = None      # TODO(you): your number, e.g. 4
METHOD = ""               # TODO(you): one line on how you chose it

submit_prediction(TEAM, 8, MY_PREDICTION, METHOD)

**Write it down** (on the sheet, or double-click here): your number, the baseline,
and the one reason you moved away from the baseline, if you did.

**Our number / the baseline / why:**

---
## Read before you run

A student replaced the analyse stage of their skeleton with this. The pipeline still ran end to end without an error:

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it.

1. **Read it. Do not run it.** Answer in the text box under the code cell
   (double-click it to type): first *what is wrong with it*, then *what it would
   produce and what it should produce*. The questions are just below.
2. **Ask Gemini to explain it.** Click once in the code cell. At its top right,
   click the icon that looks like a pencil with a small star (Colab calls it the
   Gemini icon) and choose **Explain code**. Read the explanation. Did it mention
   the fault? It often does not.
3. **Only then run it.** Delete the `# ` at the start of every line *below the
   first line* (the first line is the instruction and stays), then run the cell
   and see whether you were right.

The skeleton's `ingest()` returns ten rows with `value` 0 to 9. How many rows come out of this stage? Your debounce from Lab 7 looks at “the previous three rows”. After this change, which rows are the previous rows?

Write down the rows in and rows out, and why stage 4 can no longer tell three consecutive readings from three scattered ones.

<details><summary>Hint 1: where to look</summary>

What happens to the rows whose value is 5 or less? Then ask what "the previous row" means in the next stage.

</details>
<details><summary>Hint 2: closer to the answer</summary>

Ten rows in, four out. The rows in between are gone, so "three consecutive readings" in stage 4 now means three consecutive *flagged* readings, which is a different thing.

</details>

An explanation that reads well is not evidence the code is right; that is the
point of step 2. **To fix it afterwards:** click the same Gemini icon on the cell,
choose **Transform code**, and say what to change in plain words. Colab shows the
change as a diff before it is applied. Read the diff. If it touches a `check.` line,
reject it.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# def analyse(df):
#     return df[df.value > 5].assign(flag=True)

**What is wrong with it:**

**What it would produce, and what it should produce:**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the five stages and what passes between them; why two correct stages can fail at the join; what the row count tells you at each stage boundary; what rows_accounted_for and response_fired check end to end; where in the pipeline your project's biggest risk sits. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- The whole chain runs end to end from real data to a real notification.
- **Mistake caught** (below): which join broke first, and what was the
  mismatched assumption?
- A **Decisions** line for what passes between your stages (cadence, columns) and
  why. Before you close the notebook, ask your assistant to summarise what it
  changed today; check the summary against your row counts and correct it before
  you paste it in.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*